<a href="https://colab.research.google.com/github/akshay-aiml/LangChain_LangGraph/blob/main/coditional_node_router_function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Conditional Edges allow your agent to make decisions. Without them, your agent is just a fixed pipeline; with them, it’s a living logic flow.

1. What are Conditional Edges?
A conditional edge doesn't just point from Node A to Node B. Instead, it points from Node A to a Router Function. This function looks at the current State and returns a string that tells the graph which node to visit next.

2. The Components
To implement this, you need three things:

**The Source Node**: Where the decision starts.

**The Router Function**: A plain Python function that returns a destination key.

**The Mapping**: A dictionary that links the router's output to the actual node names.

In [ ]:
!pip install -q  langgraph langchain_core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.6 MB/s eta 0:00:00


In [ ]:
##Let's build a graph where an LLM writes an answer, and a router decides if it's "Good" (Go to END) or "Bad" (Go back to the LLM to try again).

In [ ]:
import os
from typing import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

In [ ]:
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")
# Initialize LLM

llm = ChatGroq(
    api_key=api_key,
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

In [ ]:
class AgentState(TypedDict):
    question: str
    answer: str
    score: int

In [ ]:
#Nodes

def generate_answer(state: AgentState):
    print("--- GENERATING ---")
    res = llm.invoke(state["question"])
    # Mocking a score: in reality, another LLM could score this
    score = 85 if "-" in res.content.lower() else 40
    return {"answer": res.content, "score": score}

In [ ]:
## Automatic pass after rewrite
def improve_answer(state: AgentState):
    print("--- REWRITING FOR BETTER QUALITY ---")
    res = llm.invoke(f"Improve this answer: {state['answer']}")
    return {"answer": res.content, "score": 90}

In [ ]:
# THE ROUTER FUNCTION
def route_based_on_score(state: AgentState):
    if state["score"] > 70:
        return "perfect"
    else:
        return "fail"

In [ ]:
# 4. Build Graph
workflow = StateGraph(AgentState)

workflow.add_node("writer", generate_answer)
workflow.add_node("reviser", improve_answer)

workflow.set_entry_point("writer")

In [ ]:
# 5. ADD CONDITIONAL EDGES
workflow.add_conditional_edges(
    "writer",  route_based_on_score,
  {
      "perfect": END,      # If router says 'perfect', stop
      "fail": "reviser"    # If router says 'fail', go to reviser
  }
)

In [ ]:
workflow.add_edge("reviser", END)

In [ ]:
app = workflow.compile()

In [ ]:
initial_input = {"question": "What is the capital of India ?"}

In [ ]:
final_state = app.invoke(initial_input)

--- GENERATING ---
--- REWRITING FOR BETTER QUALITY ---


In [ ]:
print(final_state["answer"])

The capital of India is indeed New Delhi, which is a part of the larger metropolitan area of Delhi. New Delhi serves as the country's administrative and governmental hub, hosting various important institutions, including the Parliament of India, the President's residence (Rashtrapati Bhavan), and numerous government ministries and departments. It is also a major cultural and economic center, known for its rich history, diverse cuisine, and vibrant cultural scene.
